In [ ]:
# src/train_baseline.py
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

def main():
    df = pd.read_parquet("data/processed/speeches_embeddings.parquet")
    df = df[df["motion_passed"].notna()].copy().sort_values("speech_begin")

    train_end = int(len(df) * 0.70)
    val_end = int(len(df) * 0.85)

    train = df.iloc[:train_end]
    test = df.iloc[val_end:]

    emb_cols = [c for c in df.columns if c.startswith("emb_")]

    num_cols = [
        "speech_duration_seconds", "hour_of_day", "day_of_week", "month",
        "hour_sin", "hour_cos",
        "sentiment_score_neg", "sentiment_score_neu", "sentiment_score_pos"
    ] + emb_cols[:128]

    cat_cols = ["party", "session_type", "time_bin", "sentiment_label"]

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
        ]
    )

    clf = Pipeline([
        ("pre", pre),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ])

    X_train = train[num_cols + cat_cols]
    y_train = train["motion_passed"]
    X_test = test[num_cols + cat_cols]
    y_test = test["motion_passed"]

    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    print(classification_report(y_test, pred))
    print("ROC-AUC:", roc_auc_score(y_test, proba))

if __name__ == "__main__":
    main()

In [ ]:
# src/train_xgboost.py
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

def main():
    df = pd.read_parquet("data/processed/speeches_embeddings.parquet")
    df = df[df["motion_passed"].notna()].copy().sort_values("speech_begin")

    train_end = int(len(df) * 0.70)
    val_end = int(len(df) * 0.85)

    train = df.iloc[:train_end]
    test = df.iloc[val_end:]

    emb_cols = [c for c in df.columns if c.startswith("emb_")]

    num_cols = [
        "speech_duration_seconds", "hour_of_day", "day_of_week", "month",
        "hour_sin", "hour_cos",
        "sentiment_score_neg", "sentiment_score_neu", "sentiment_score_pos"
    ] + emb_cols

    cat_cols = ["party", "session_type", "time_bin", "sentiment_label"]

    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ])

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc",
        tree_method="hist"
    )

    pipe = Pipeline([("pre", pre), ("model", model)])

    X_train = train[num_cols + cat_cols]
    y_train = train["motion_passed"]
    X_test = test[num_cols + cat_cols]
    y_test = test["motion_passed"]

    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    print(classification_report(y_test, pred))
    print("ROC-AUC:", roc_auc_score(y_test, proba))

if __name__ == "__main__":
    main()